[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pinecone-io/examples/blob/master/learn/generation/langchain/handbook/05-langchain-retrieval-augmentation.ipynb) [![Open nbviewer](https://raw.githubusercontent.com/pinecone-io/examples/master/assets/nbviewer-shield.svg)](https://nbviewer.org/github/pinecone-io/examples/blob/master/learn/generation/langchain/handbook/05-langchain-retrieval-augmentation.ipynb)

#### [LangChain Handbook](https://www.pinecone.io/learn/series/langchain/)

# Retrieval Augmentation

**L**arge **L**anguage **M**odels (LLMs) have a data freshness problem. Even the most powerful LLMs in the world are not up to date with recent world events.

The world of LLMs is frozen in time. Their world exists as a static snapshot of the world as it was within their training data.

A solution to this problem is *retrieval augmentation*. The idea behind this is that we retrieve relevant information from an external knowledge base and give that information to our LLM. In this notebook we will learn how to do that.

[![Open fast notebook](https://raw.githubusercontent.com/pinecone-io/examples/master/assets/fast-link.svg)](https://colab.research.google.com/github/pinecone-io/examples/blob/master/generation/langchain/handbook/05-langchain-retrieval-augmentation-fast.ipynb)

To begin, we must install the prerequisite libraries that we will be using in this notebook.

In [ ]:
# Use %pip (not !pip) so packages install into THIS notebook's kernel,
# not some other Python on your system. Restart the kernel after this runs.
%pip install -qU     datasets==3.6.0     langchain==0.3.25     langchain-community==0.3.25     langchain-google-genai==2.0.10     langchain-qdrant==0.2.1     qdrant-client     sniffio anyio pyparsing     tiktoken==0.9.0

# --- OpenRouter alternative (for reference) ---
# %pip install -qU langchain-openai==0.3.22

## Building the Knowledge Base

In [ ]:
from datasets import load_dataset

data = load_dataset(
    "aurelio-ai/reddit-finance",
    split="train",
)
data

In [ ]:
data[5]

Many records contain *a lot* of text. Our first task is therefore to identify a good preprocessing methodology for chunking these articles into more "concise" chunks to later be embedding and stored in our Qdrant vector database.

For this we use LangChain's `RecursiveCharacterTextSplitter` to split our text into chunks of a specified max length.

In [ ]:
import tiktoken

# We use gpt-4.1-mini as standard but tiktoken does not support gpt-4.1.
# Fortunately, 4.1 and 4o models all use the same underlying tokenizer and so
# we can use gpt-4o here
encoding = tiktoken.encoding_for_model('gpt-4o')

The tokenizer encoding that we'll use is:

In [ ]:
encoding.name

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding(encoding.name)

# create the length function
def tiktoken_len(text):
    tokens = tokenizer.encode(
        text,
        disallowed_special=()
    )
    return len(tokens)

tiktoken_len("hello I am a chunk of text and using the tiktoken_len function "
             "we can find the length of this chunk of text in tokens")

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=20,
    length_function=tiktoken_len,
    separators=["\n\n", "\n", " ", ""]
)

In [ ]:
chunks = text_splitter.split_text(data[5]['selftext'])
chunks

In [ ]:
tiktoken_len(chunks[0]), tiktoken_len(chunks[1])

Using the `text_splitter` we get much better sized chunks of text. We'll use this functionality during the indexing process later. Now let's take a look at embedding.

## Creating Embeddings

Building embeddings using LangChain's Google Gemini embedding support is fairly straightforward. We first need to add our [Google API key](https://aistudio.google.com/app/apikey) by running the next cell:

In [ ]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")     or getpass("Enter your Google API key: ")

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# --- OpenRouter alternative (for reference) ---
# os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY") or getpass("Enter your OpenRouter API key: ")
# OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

After initializing the API key we can initialize our `gemini-embedding-001` embedding model like so:

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

model_name = 'models/gemini-embedding-001'

embed = GoogleGenerativeAIEmbeddings(
    model=model_name,
    google_api_key=GOOGLE_API_KEY
)

# --- OpenAI alternative (for reference) ---
# from langchain_openai import OpenAIEmbeddings
# embed = OpenAIEmbeddings(
#     model='text-embedding-3-small',
#     openai_api_key=OPENAI_API_KEY
# )

Now we embed some text like so:

In [ ]:
texts = [
    'this is the first chunk of text',
    'then another second chunk of text is here'
]

res = embed.embed_documents(texts)
len(res), len(res[0])

From this we get *two* (aligning to our two chunks of text) 3072-dimensional embeddings.

Now we move on to initializing our Qdrant vector database.

## Vector Database

We will connect to an existing Qdrant deployment. No API key is required for this deployment, we only need the URL.

In [ ]:
# Qdrant here needs no API key, we only need the URL of the running deployment.
QDRANT_URL = "http://109.199.118.73:6333"

Then we create the collection. We will be using Google's `gemini-embedding-001` model for creating the embeddings, so we set the vector `size` to `3072` and use cosine distance.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

client = QdrantClient(url=QDRANT_URL)
collection_name = "handbook-05-rag"

if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE),
    )

# view collection info
client.get_collection(collection_name)

We should see that the new Qdrant collection has a `points_count` of `0`, as we haven't added any vectors yet.

## Indexing

We can perform the indexing task using the LangChain `QdrantVectorStore` object. The vector store embeds the texts via the Gemini `embed` model automatically and upserts them into the collection. Let's first initialize the vector store.

In [ ]:
from langchain_qdrant import QdrantVectorStore

# initialize the vector store object
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embed,
)

from tqdm.auto import tqdm

batch_limit = 100

texts = []
metadatas = []

for i, record in enumerate(tqdm(data)):
    # first get metadata fields for this record
    url = f"https://reddit.com/r/{record['subreddit']}/comments/{record['id']}"
    metadata = {
        'thread_id': str(record['id']),
        'source': url,
        'subreddit': record['subreddit']
    }
    # now we create chunks from the record text
    record_texts = text_splitter.split_text(record['selftext'])
    # create individual metadata dicts for each chunk
    record_metadatas = [{
        "chunk": j, "text": text, **metadata
    } for j, text in enumerate(record_texts)]
    # append these to current batches
    texts.extend(record_texts)
    metadatas.extend(record_metadatas)
    # if we have reached the batch_limit we can add texts
    if len(texts) >= batch_limit:
        vectorstore.add_texts(texts=texts, metadatas=metadatas)
        texts = []
        metadatas = []

if len(texts) > 0:
    vectorstore.add_texts(texts=texts, metadatas=metadatas)

We've now indexed everything. We can check the number of vectors in our collection like so:

In [ ]:
client.get_collection(collection_name)

## Querying

Now that we've built our collection we can query the vector store directly. The `vectorstore` object we created above is already connected to the Qdrant collection, so we can search it like so:

In [ ]:
# The `vectorstore` object created during indexing is already connected to our
# Qdrant collection. If you are reconnecting in a fresh session, you can rebuild
# it from the existing collection like so:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embed,
)

In [ ]:
query = "how many robotaxi rides did waymo report in the US?"

vectorstore.similarity_search(
    query,  # our search query
    k=3  # return 3 most relevant docs
)

All of these are good, relevant results. But what can we do with this? There are many tasks, one of the most interesting (and well supported by LangChain) is called _"Retrieval Augmented Generation"_ or RAG.

## Retrieval Augmented Generation

In RAG we take the query as a question that is to be answered by a LLM, but the LLM must answer the question based on the information it is seeing being returned from the `vectorstore`.

To do this we initialize a retrieval pipeline like so:

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# completion llm
llm = ChatGoogleGenerativeAI(
    temperature=0.0,
    google_api_key=GOOGLE_API_KEY,
    model="gemini-2.5-flash",
)

# --- OpenRouter alternative (for reference) ---
# OpenRouter is OpenAI-compatible: use the ChatOpenAI class with a different base_url.
# Pick any ":free" model from https://openrouter.ai/models, e.g. "openai/gpt-oss-120b:free".
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(
#     model="openai/gpt-oss-120b:free",
#     temperature=0.0,
#     api_key=OPENROUTER_API_KEY,
#     base_url="https://openrouter.ai/api/v1",
# )

# Create prompt template
template = """Answer the question based on the following context:

{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

# Create LCEL chain
retrieval_chain = (
    {"context": vectorstore.as_retriever(), "question": lambda x: x}
    | prompt
    | llm
    | StrOutputParser()
)

print(query)

retrieval_chain.invoke(query)

We can also include the sources of information that the LLM is using to answer our question:

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Create prompt template with source formatting
template = """Answer the question based on the following context. Include the source URLs in your answer.

{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

# Create LCEL chain with source formatting
def format_docs(docs):
    return "\\n\\n".join([f"Source: {doc.metadata.get('source', 'Unknown')}\\n{doc.page_content}" for doc in docs])

retrieval_chain_with_sources = (
    {"context": vectorstore.as_retriever() | format_docs, "question": lambda x: x}
    | prompt
    | llm
    | StrOutputParser()
)

print(f"Question: {query}\n")
print(retrieval_chain_with_sources.invoke(query))

Now we answer the question being asked, *and* return the source of this information being used by the LLM.

Delete the collection to save resources when you're done!

In [ ]:
client.delete_collection(collection_name)

---